# Building Next-Generation RAG with LangChain

**Advanced NLP Course — Exercise Session**  
Leibniz Universität Hannover

This notebook shows a transition from Naive RAG to an advanced RAG pipeline featuring a multi-retriever Ensemble (Hybrid Search) and an explicit context structure using LangChain Expression Language (LCEL).

## Part 1: Introduction & Basic Naive RAG with LangChain

In this section, we build a foundational Naive RAG chain using LangChain's decoupled components: ``VectorStore ➔ PromptTemplate ➔ LLM ➔ StrOutputParser```.

**Step 1. Install LangChain & Core Dependencies**

In [ ]:
!pip install langchain langchain-core langchain-community -q
!pip install faiss-cpu sentence-transformers langchain-huggingface -q

**Step 2. Define a Mock Knowledge Base (Corpus)**

LangChain uses native ``Document`` objects wrapped with ``page_content`` and ``metadata`` dictionaries.

In [ ]:
from langchain_core.documents import Document
import json

with open("documents.json", "r") as file:
    data = json.load(file)['documents']

documents = [Document(page_content=json.dumps(d), metadata={'id':idx})
             for idx, d in enumerate(data)]

**Step 3. Initialize the Vector Store & Embedding Model**

We initialize a local, in-memory Faiss vector store using a lightweight embedding model.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Initialize local lightweight embedding model
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

# 2. Populate Faiss Vector DB
vectorstore = FAISS.from_documents(documents, embeddings)

# 3. Create a basic retriever interface exposing top 2 docs
naive_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

**Step 4. Construct and Run the Naive LCEL Chain**

We stitch the architecture together sequentially using the ``|`` operator (LangChain Expression Language).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline as hf_pipeline

# 1. Setup a lightweight local open-source LLM
hf_pipe = hf_pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", max_new_tokens=50, temperature=0.2)
llm = HuggingFacePipeline(pipeline=hf_pipe)

# 2. Structure standard context injection prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
Answer:""")

# 3. Helper function to clean up retrieved document lists into a single string block
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Construct Naive RAG Chain
naive_rag_chain = (
    {"context": naive_retriever | format_docs, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

query = "What is the main purpose of the LangChain framework?"
response = naive_rag_chain.invoke(query)

print(f"Query: {query}\n")
print(f"Generated Response:\n{response}")

## Part 2: Real-World Scenario – Advanced Hybrid Retrieval & Ensemble

**Scenario:**  Your application needs both exact technical keyword hits and semantic match captures. To scale beyond simple vector match flaws, we will construct an Advanced RAG hybrid architecture inside LangChain utilizing an ``EnsembleRetriever``. This aggregates predictions from both a BM25 Keyword Retriever and a Dense Semantic Retriever via Reciprocal Rank Fusion (RRF).

A workflow:
```
                       ┌──> Dense Retriever (FAISS) ──────┐
Query ──> LCEL Chain ──┤                                  ├──> Ensemble (RRF Merge) ──> LLM Prompt
                       └──> Sparse Retriever (BM25)  ─────┘
```

**Step 1. Instantiate the Advanced Hybrid Retrievers**

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# 1. Create a traditional keyword sparse retriever from the documents
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 2

# 2. Keep our dense vector retriever from Part 1
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 3. Create an Advanced Ensemble Retriever combining both using RRF
# weights=[0.5, 0.5] applies equal ranking influence to keyword vs semantic matches
advanced_ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever], 
    weights=[0.5, 0.5]
)

**Step 2. Assemble and Evaluate the Complex Production Chain**

We swap our retrieval layer inside the execution sequence cleanly without modifying our downstream prompt configurations.

In [ ]:
# 1. Advanced Chain Construction
advanced_rag_chain = (
    {"context": advanced_ensemble_retriever | format_docs, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

# 2. Run a complex query requiring both architectural paths
complex_query = "Explain advanced RAG systems in the current year 2026."
output = advanced_rag_chain.invoke(complex_query)

print(f"Resulting Answer:\n{output}")

---
*Advanced NLP · Exercise Session · SoSe 2026*